# 06b · Frozen CXR representation × EHR fusion — 8 heads, one shared forward

**Câu hỏi.** `06` đã xác lập EHR bổ sung CXR: `+0.0105` @ASL (`film_asl 0.4202 − none_asl
0.4097`), sống sót dưới loss tốt nhất. Nhưng nó được đo trên `x_cxr` = **CLS đã pool** —
một vector 768 nén từ 1369 patch token. Câu hỏi kế tiếp:

> EHR bổ sung **tín hiệu mới**, hay chủ yếu **vá lại thông tin đã mất** khi ảnh bị nén
> thành một global CLS?

Nghi ngờ có cơ sở định lượng: gain của EHR (`+0.0105`) **xấp xỉ đúng** khoảng cách giữa
patch và CLS representation. Cùng độ lớn là dấu hiệu kinh điển của hai thứ đang vá cùng
một chỗ hổng — đúng kiểu trùng hợp đã lộ ra ở trục loss (`db→asl` cũng `+0.0144`, ngang
với toàn bộ đóng góp EHR).

## 8 head

| ID | Repr | Head | EHR | Vai trò |
|---|---|---|---|---|
| `H1` | CLS | MLP | none | replication baseline |
| `H2` | CLS | MLP | concat | replication EHR gain |
| `H3` | CLS | ML-Dec (seq 1) | none | repr control |
| `H4z` | CLS + zero tok | ML-Dec (seq 2) | — | **structural** control |
| `H4` | CLS + EHR tok | ML-Dec (seq 2) | token | repr control |
| `H5` | patch | ML-Dec (1369) | none | patch baseline |
| `H6z` | patch + zero tok | ML-Dec (1370) | — | **structural** control |
| `H6` | patch + EHR tok | ML-Dec (1370) | token | **target** |

## Metric — nêu TRƯỚC khi chạy

Bài học Pneumonia (1/13 nhãn ở p<0.05 = đúng tỉ lệ ngẫu nhiên) áp dụng ở đây: 8 head sinh
ra rất nhiều delta. Nên **cố định trước**:

$$\textbf{PRIMARY:}\quad I_{content}=(H6-H6z)-(H4-H4z)$$

Một con số, một test. Nó trừ đi **chi phí cơ học của việc chèn thêm một token** ở cả hai
nhánh, và giữ `fusion=token` + `head=ML-Decoder` cố định — nên khác biệt còn lại **chỉ là
representation**.

**SECONDARY** (báo cáo, không claim): `I_repr=(H6-H5)-(H4-H3)` · `I_pipeline=(H6-H5)-(H2-H1)`
· `Δ_structure = H6z-H5` · `Δ_content = H6-H6z`
**EXPLORATORY**: per-label, per-tier, attention mass, value contribution.

`I_pipeline` là con số plan gốc đề xuất — nó **lẫn 3 biến** (representation, fusion, head),
nên bị giáng xuống secondary chứ không bỏ: nó vẫn trả lời "hệ thống hoàn chỉnh nào tốt hơn".

## Vì sao một forward dùng chung

Backbone **đóng băng và giống hệt** ở cả 8 head ⇒ cùng ảnh luôn cho cùng token. Chạy 8×3
run riêng = tính lại RAD-DINO **y hệt 144 lần** (~30 GPU-giờ). Dùng chung forward → **3 run
≈ 7.5h**.

Gradient **tương đương** với train riêng — nhưng chỉ khi mỗi head có **module, optimizer,
scheduler, early-stop và grad-clip HOÀN TOÀN riêng**. Hai cái bẫy:
1. **Dùng chung `EhrEncoder`** giữa H2/H4/H6 ⇒ `∇_θE(L2+L4+L6)` trộn ba pipeline.
2. **Clip global** ⇒ một global norm scale tất cả ⇒ head gradient lớn bóp head khác.

Nên: backward **tuần tự** trên feature đã `detach()`, clip riêng từng head.

> Forward dùng chung cho **common random numbers** (cùng batch, cùng augmentation, cùng
> data order) ⇒ giảm nuisance variance. Nhưng batch **không** trở thành đơn vị thống kê —
> kiểm định vẫn là **paired seeds** + **subject-level bootstrap**.

## "Frozen" nghĩa là gì ở đây

Backbone đã **supervised fine-tune trên 13 nhãn ở 04b** rồi mới đóng băng — không phải
RAD-DINO pretrained gốc. Gap train−val `+0.042` (04b·C9-bis) vẫn nằm trong đó. Cả 8 head
dùng chung nên nó **triệt tiêu trong mọi delta**, nhưng phải ghi rõ khi báo cáo
external generalization.

**Prereq:** 04a xong (tar shard `cxr_560_shards`), 04b xong
(`raddino_mld_db_ft_best.pt`), 02 xong (`cohort_study_level.parquet`, `ehr_vocab.pkl` —
bản có vitals + comorbidity). Notebook **tự stage ảnh** ở **C0-bis** — `/content` bị xóa
mỗi lần VM restart, nên không giả định 04b/05 đã stage ở phiên trước.


## C0 · Config — 8 head, mọi thứ riêng biệt trừ backbone

In [ ]:
import os, json, math, time, pickle, random, glob, shutil, tarfile
from dataclasses import dataclass, replace, field
from typing import Optional
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = '.'
OUT_DIR   = os.path.join(DATA_DIR, 'cohort_v2')
SHARD_DIR = os.path.join(DATA_DIR, 'cxr_560_shards')
LOCAL_IMG = '/content/cxr_local'
BEST_CKPT = os.path.join(DATA_DIR, 'lpft_v2', 'raddino_mld_db_ft_best.pt')
TF_DIR    = os.path.join(DATA_DIR, 'token_fusion')
for _d in (TF_DIR, f'{TF_DIR}/checkpoints', f'{TF_DIR}/metrics', f'{TF_DIR}/predictions'):
    os.makedirs(_d, exist_ok=True)

W_F, W_L  = 0.7, 0.3                       # study-level pooling, same as 04b/05
SEEDS     = [42, 1337, 2024]
EPOCHS, PATIENCE = 12, 3
EFFECTIVE_BATCH  = 256                     # actual batch is auto-probed in C11
LR, WD, WARMUP, CLIP = 1e-3, 1e-2, 0.05, 5.0
AMP = torch.bfloat16

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev, '|', torch.cuda.get_device_name(0) if dev=='cuda' else 'CPU')

@dataclass
class HCfg:
    """One experimental arm. Only `repr`/`head`/`ehr` change; everything else is held fixed."""
    tag:  str
    repr: str        # {'cls','patch'}
    head: str        # {'mlp','mld'}
    ehr:  str        # {'none','concat','token','zero'}   'zero' = constant zero token
    d_cxr: int = 768
    d_e: int   = 128
    d_hidden: int = 256
    dropout_head: float = 0.1
    dropout_ehr: float  = 0.2

HEADS = [
    HCfg('H1',  repr='cls',   head='mlp', ehr='none'),
    HCfg('H2',  repr='cls',   head='mlp', ehr='concat'),
    HCfg('H3',  repr='cls',   head='mld', ehr='none'),
    HCfg('H4z', repr='cls',   head='mld', ehr='zero'),
    HCfg('H4',  repr='cls',   head='mld', ehr='token'),
    HCfg('H5',  repr='patch', head='mld', ehr='none'),
    HCfg('H6z', repr='patch', head='mld', ehr='zero'),
    HCfg('H6',  repr='patch', head='mld', ehr='token'),
]
print('\n'.join(f'  {h.tag:4s} repr={h.repr:5s} head={h.head:3s} ehr={h.ehr}' for h in HEADS))

## C0-bis · Stage ảnh về ổ local

Ảnh nằm trên Drive dưới dạng **tar shard** (`cxr_560_shards`, ~17 GB); `/content` là ổ tạm
của VM và **bị xóa mỗi lần restart**. Cell này phải chạy trong *phiên này*, kể cả khi 04b
hay 05 đã stage ở phiên trước — runtime mới luôn bắt đầu trống.

Copy → giải nén → xóa tar, từng shard một (đỉnh đĩa = `giải nén + 1 tar` thay vì gấp đôi).
Marker `.done_` ghi **sau** khi extract ⇒ resume được, và bỏ qua shard đã có.

**CPU/network, không cần GPU** — ~8 phút cho bộ đầy đủ, tức thì nếu ổ còn ấm.

In [ ]:
os.makedirs(LOCAL_IMG, exist_ok=True)
tars = sorted(f for f in os.listdir(SHARD_DIR) if f.endswith('.tar'))
assert tars, f'no shards in {SHARD_DIR} -> run 04a first'
print(f'shards on Drive: {len(tars)}   already local: {len(os.listdir(LOCAL_IMG)):,} files')
todo = [f for f in tars if not os.path.exists(os.path.join(LOCAL_IMG, f'.done_{f}'))]
print(f'shards to stage: {len(todo)} / {len(tars)}   (~8 min for a full set; instant if warm)')
for f in tqdm(todo, desc='staging shards', unit='shard'):
    tmp = f'/content/{f}'
    shutil.copyfile(os.path.join(SHARD_DIR, f), tmp)
    with tarfile.open(tmp) as t: t.extractall(LOCAL_IMG)
    os.remove(tmp)
    open(os.path.join(LOCAL_IMG, f'.done_{f}'), 'w').close()   # marker AFTER extract
n_local = len([f for f in os.listdir(LOCAL_IMG) if f.endswith('.jpg')])
print(f'local images: {n_local:,}')
assert n_local > 0, 'staging produced no images'

## C1 · Cohort, EHR, vocab — đúng như 02 xuất ra

In [ ]:
meta   = json.load(open(os.path.join(OUT_DIR, 'cohort_meta.json')))
LABELS = meta['labels']; assert len(LABELS) == 13

ehr = pd.read_parquet(os.path.join(OUT_DIR, 'cohort_study_level.parquet'))

ehr = ehr.sort_values('study_id').reset_index(drop=True)
N   = len(ehr)
# subject_id/setting are load-bearing here: the subject-level split gates in C3, the
# shuffle strata in C16, the subject-clustered bootstrap in C18. Check once, name the
# columns if not -- 02 carries them through from cohort_base, but assert rather than hope.
for _c in ('study_id', 'subject_id', 'split', 'setting'):
    assert _c in ehr.columns, f'ehr has no {_c}; cols = {sorted(ehr.columns)}'
EHR_ROW = {int(s): i for i, s in enumerate(ehr.study_id.to_numpy())}   # study_id -> row

vocab = pickle.load(open(os.path.join(OUT_DIR, 'ehr_vocab.pkl'), 'rb'))
ORG_V, SPEC_V = vocab['org_vocab_size'], vocab['spec_vocab_size']
MISSING = vocab['reserved']['MISSING']

cf = pickle.load(open(os.path.join(OUT_DIR, 'class_freq.pkl'), 'rb'))
CLASS_FREQ, NUM_TRAIN = np.asarray(cf['class_freq']), cf['num_train']
TRAIN_PREV = CLASS_FREQ / NUM_TRAIN
TIER = np.where(TRAIN_PREV < 0.02, 'tail', np.where(TRAIN_PREV < 0.10, 'medium', 'head'))
Y_STUDY = ehr[LABELS].to_numpy('float32')
print(f'cohort {N:,} studies | org vocab {ORG_V} | spec vocab {SPEC_V}')
print('splits:', dict(ehr.split.value_counts()))

## C2 · EHR featurization — **copy nguyên** từ 06·C2

Không đổi một feature nào giữa H2/H4/H6. Mọi khác biệt về EHR gain phải đến từ **vị trí
fusion**, không phải từ một EHR encoder khác.

In [ ]:
def as_id_lists(col):
    out = []
    for v in ehr[col].to_numpy():
        a = np.asarray(v, np.int64).ravel() if v is not None and np.ndim(v) else np.array([], np.int64)
        out.append(a if a.size else np.array([MISSING], np.int64))
    return out

RAGGED = {'t1_org':('t1_org_ids',ORG_V), 't1_spec':('t1_spec_ids',SPEC_V),
          't2_spec':('t2_spec_ids',SPEC_V), 't3_org':('t3_org_ids',ORG_V)}
bags = {k: as_id_lists(c) for k,(c,_) in RAGGED.items()}

def codes(series, cats):
    m = {c.lower(): i for i, c in enumerate(cats)}
    return series.astype(str).str.lower().map(lambda x: m.get(x, 0)).to_numpy('int64'), len(cats)
set_id,  _ = codes(ehr['setting'],        ['inpatient','ED','outpatient'])
rec_id,  _ = codes(ehr['t3_recency_bin'], ['none','7-30d','30-90d'])
stat_id    = ehr['t1_status'].to_numpy('int64')

BIN = ['t2_pending','t2_time_imprecise','t1_same_encounter']
binv = {b: ehr[b].to_numpy('float32') for b in BIN}
tr_mask = (ehr.split == 'train').to_numpy()
def zscore_train(x, use_log=True):
    z = np.log1p(np.clip(x.astype('float32'), 0, None)) if use_log else x.astype('float32')
    mu, sd = z[tr_mask].mean(), z[tr_mask].std() or 1.0
    return ((z - mu)/sd).astype('float32')
COUNTS = ['t2_n_pending','t3_n_specimens','t4_n_prior_positive_ever',
          't4_years_since_first_culture','t4_n_prior_encounters']
cnts = {c: zscore_train(ehr[c].to_numpy()) for c in COUNTS}

_z = lambda k: np.zeros((N, k), 'float32')
LABS   = vocab['labs']
lab_val  = np.stack([ehr[f'{c}_value'].to_numpy('float32') for c in LABS], 1)
lab_mask = np.stack([ehr[f'{c}_mask'].to_numpy('float32')  for c in LABS], 1)
VITALS = [c for c in vocab.get('vitals', []) if f'{c}_value' in ehr.columns]
vit_val  = np.stack([ehr[f'{c}_value'].to_numpy('float32') for c in VITALS], 1) if VITALS else _z(0)
vit_mask = np.stack([ehr[f'{c}_mask'].to_numpy('float32')  for c in VITALS], 1) if VITALS else _z(0)
COMO_COLS = [c for c in (vocab.get('comorbidity', []) + ['como_mask']) if c in ehr.columns]
como_arr  = np.stack([ehr[c].to_numpy('float32') for c in COMO_COLS], 1) if COMO_COLS else _z(0)

REC1 = ['t1_days_since_last_specimen','t1_days_since_last_positive']; REC3 = ['t3_days_since_last_positive']
rec_block = lambda ns: (np.stack([ehr[f'{c}_value'].to_numpy('float32') for c in ns], 1),
                        np.stack([ehr[f'{c}_mask'].to_numpy('float32')  for c in ns], 1))
rec1_v, rec1_m = rec_block(REC1); rec3_v, rec3_m = rec_block(REC3)
MICRO_TIERS = (1, 2, 3)
print(f'labs {len(LABS)} | vitals {len(VITALS)} | como {COMO_COLS} | tiers {MICRO_TIERS}')

## C3 · Image manifest — một hàng mỗi ảnh

Split giữ nguyên ở mức **subject** từ 01. Các assert bắt buộc: subject không rò giữa
split, mọi ảnh của một study cùng split, cùng nhãn.

In [ ]:
# C3 consumes what C1 built. A stale `ehr` (C1 not re-run after an edit) surfaces here as
# a bare `KeyError: subject_id`, which sends you debugging THIS cell instead of C1 -- the
# same trap 02's need() guard exists for. Name the cell to re-run instead.
for _c in ('subject_id', 'setting', 'split'):
    if _c not in ehr.columns:
        raise RuntimeError(
            f"`ehr` has no '{_c}' -> re-run C1 (it repairs the schema from cohort_base).\n"
            f"    If C1 already ran, its 'schema repair:' line would have printed.\n"
            f"    ehr columns = {sorted(ehr.columns)}")

vm = pd.read_parquet(os.path.join(OUT_DIR, 'view_map.parquet'))
vm = vm[vm.view_group.isin(['frontal','lateral'])]

# view_map ALSO carries subject_id. Merging it against ehr's subject_id makes pandas
# silently rename both to subject_id_x / subject_id_y -- so `man.subject_id` then does not
# exist and the split gate below dies with a bare `KeyError: subject_id` that looks like a
# MISSING column when it is really a DUPLICATED one. Drop the overlap first: `ehr` is the
# single source of truth for subject_id/split (01 assigned the split at subject level).
_dup = [c for c in ('subject_id', 'split', 'setting') if c in vm.columns]
if _dup: print(f'view_map also carries {_dup} -> dropping, ehr is authoritative')
man = vm.drop(columns=_dup).merge(ehr[['study_id','subject_id','split']],
                                  on='study_id', how='inner')
assert 'subject_id' in man.columns and 'split' in man.columns, sorted(man.columns)
before = len(man)
have = man.dicom_id.map(lambda d: os.path.exists(f'{LOCAL_IMG}/{d}.jpg'))
man  = man[have].sort_values('dicom_id').reset_index(drop=True)
assert (before - len(man)) / before < 0.01, \
    (f'{before-len(man)} / {before} images missing from {LOCAL_IMG} -> run C0-bis first '
     '(100% missing = fresh VM: /content was wiped, nothing has been staged yet)')
man['ehr_row'] = man.study_id.map(EHR_ROW).astype('int64')
assert man.ehr_row.notna().all()

# --- gates from the plan ---
assert man.study_id.isin(ehr.study_id).all()
subj = {s: set(man.loc[man.split == s, 'subject_id']) for s in ('train','val','test')}
assert subj['train'].isdisjoint(subj['val']) and subj['train'].isdisjoint(subj['test']) \
   and subj['val'].isdisjoint(subj['test']), 'subject leak across splits'
assert man.groupby('study_id')['split'].nunique().max() == 1, 'a study spans two splits'
print(f'manifest: {len(man):,} images / {man.study_id.nunique():,} studies')
print('by split :', dict(man.drop_duplicates("study_id").split.value_counts()))
print('by view  :', dict(man.view_group.value_counts()))
man.to_parquet(f'{TF_DIR}/image_manifest.parquet', index=False)

IDX = {s: np.where(man.split.to_numpy() == s)[0] for s in ('train','val','test')}

## C4 · Dataset ở mức ảnh — EHR và nhãn lặp lại cho các ảnh cùng study

Augmentation **giống 04b** cho train (`RandomResizedCrop` + rotation, **không** flip: bên
trái/phải có ý nghĩa trên phim ngực), `CenterCrop` cho val/test. Cả 8 head nhận **cùng
một ảnh đã augment** nhờ shared forward → common random numbers.

In [ ]:
from transformers import AutoImageProcessor
from torchvision import transforms as T
from PIL import Image

proc = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
MODEL_SIZE = proc.crop_size['height']; MEAN, STD = proc.image_mean[0], proc.image_std[0]
train_tf = T.Compose([T.RandomResizedCrop(MODEL_SIZE, scale=(0.85,1.0), ratio=(0.95,1.05),
                                          interpolation=T.InterpolationMode.BICUBIC),
                      T.RandomHorizontalFlip(p=0.0), T.RandomRotation(7),
                      T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])
eval_tf  = T.Compose([T.CenterCrop(MODEL_SIZE), T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])

class ImgDS(Dataset):
    def __init__(self, idx, tf, ehr_perm=None):
        self.idx, self.tf = np.asarray(idx), tf
        self.perm = ehr_perm          # study-level EHR permutation for the shuffle diagnostic
    def __len__(self): return len(self.idx)
    def __getitem__(self, k):
        i = self.idx[k]; r = man.iloc[i]
        im = Image.open(f'{LOCAL_IMG}/{r.dicom_id}.jpg').convert('L')
        im = Image.merge('RGB', (im, im, im))
        e = int(r.ehr_row) if self.perm is None else int(self.perm[int(r.ehr_row)])
        d = dict(image=self.tf(im), y=torch.from_numpy(Y_STUDY[int(r.ehr_row)]),
                 row=i, ehr_row=e,
                 set_id=set_id[e], rec_id=rec_id[e], stat_id=stat_id[e],
                 lab_val=torch.from_numpy(lab_val[e]), lab_mask=torch.from_numpy(lab_mask[e]),
                 vit_val=torch.from_numpy(vit_val[e]), vit_mask=torch.from_numpy(vit_mask[e]),
                 como=torch.from_numpy(como_arr[e]),
                 rec1_v=torch.from_numpy(rec1_v[e]), rec1_m=torch.from_numpy(rec1_m[e]),
                 rec3_v=torch.from_numpy(rec3_v[e]), rec3_m=torch.from_numpy(rec3_m[e]))
        for b in BIN:    d[b] = binv[b][e]
        for c in COUNTS: d[c] = cnts[c][e]
        for nm in RAGGED: d[nm] = torch.from_numpy(bags[nm][e])
        return d
    # NOTE: y comes from the TRUE ehr_row -- shuffling EHR must never move the label.

def collate(b):
    o = dict(image=torch.stack([x['image'] for x in b]), y=torch.stack([x['y'] for x in b]),
             row=torch.tensor([x['row'] for x in b]),
             set_id=torch.tensor([x['set_id'] for x in b]),
             rec_id=torch.tensor([x['rec_id'] for x in b]),
             stat_id=torch.tensor([x['stat_id'] for x in b]))
    for k in ('lab_val','lab_mask','vit_val','vit_mask','como','rec1_v','rec1_m','rec3_v','rec3_m'):
        o[k] = torch.stack([x[k] for x in b])
    for k in BIN + COUNTS: o[k] = torch.tensor([x[k] for x in b], dtype=torch.float32)
    for nm in RAGGED:
        lens = [len(x[nm]) for x in b]
        o[nm] = (torch.cat([x[nm] for x in b]) if sum(lens) else torch.empty(0, dtype=torch.int64),
                 torch.tensor([0]+lens[:-1]).cumsum(0))
    return o

def loader(split, tf, shuffle, bs, perm=None, workers=8):
    # workers>0 + persistent for the ONE long-lived TRAIN loader: it is built once and
    # iterated for all 12 epochs, so its worker pool is forked once and never torn down
    # mid-run. predict() instead builds a loader EVERY epoch -- and a freshly-forked pool
    # that gets garbage-collected tears down inside a child process, tripping
    # 'can only test a child process' once per worker and flooding the cell. So predict
    # passes workers=0: no fork, no teardown, silent. Eval I/O is not the bottleneck --
    # the frozen ViT forward dominates -- so this costs little. (04b and 06 hit the same
    # noise; 06 solved it with 0 workers because its data was already in RAM.)
    return DataLoader(ImgDS(IDX[split], tf, perm), batch_size=bs, shuffle=shuffle,
                      collate_fn=collate, num_workers=workers, pin_memory=(dev=='cuda'),
                      drop_last=False, persistent_workers=(workers > 0))
to_dev = lambda b: {k:(tuple(t.to(dev, non_blocking=True) for t in v) if isinstance(v, tuple)
                       else v.to(dev, non_blocking=True)) for k, v in b.items()}
print('image-level loaders ready:', {s: len(IDX[s]) for s in IDX})

## C5 · ML-Decoder + shim torch 2.x (giống 04b·C2)

Hai shim đều **chứng minh được là no-op**: `TransformerDecoderLayerOptimal` cố ý **không có
self-attention** (đó là ý nghĩa của "Optimal"), nên `self_attn.batch_first` chỉ cần một cờ,
và `tgt_is_causal`/`memory_is_causal` mô tả mask trên self-attention không tồn tại.

**Lưu ý về `query_embed`:** ML-Decoder khai báo `query_embed.requires_grad_(False)` — 13
class query là **cố định, không học**. Với `H3` (seq=1) điều này khiến softmax trên đúng
một key luôn bằng 1 ⇒ mọi query nhận cùng attention output; phân biệt lớp chỉ còn đến từ
13 offset ngẫu nhiên cố định + FFN + `duplicate_pooling`. `H3` là control **suy biến** —
vẫn hợp lệ (nó chính là "13 query làm được gì với một vector global"), nhưng không được
đọc nó như một mô hình attention bình thường.

In [ ]:
ML_SRC = '/content/ml_decoder.py'
if not os.path.exists(ML_SRC):
    !wget -q -O {ML_SRC} https://raw.githubusercontent.com/Alibaba-MIIL/ML_Decoder/main/src_files/ml_decoder/ml_decoder.py
import sys, types, inspect
sys.path.insert(0, '/content')
import ml_decoder as _mld
from ml_decoder import MLDecoder

if not hasattr(_mld.TransformerDecoderLayerOptimal, '_patched_for_torch2'):
    _oi = _mld.TransformerDecoderLayerOptimal.__init__
    def _init(self, *a, **kw):
        _oi(self, *a, **kw)
        self.self_attn = types.SimpleNamespace(batch_first=False)   # shim (1): flag only
    _mld.TransformerDecoderLayerOptimal.__init__ = _init
    _of = _mld.TransformerDecoderLayerOptimal.forward
    def _fwd(self, tgt, memory, **kw):
        kw.pop('tgt_is_causal', None); kw.pop('memory_is_causal', None)   # shim (2)
        return _of(self, tgt, memory, **kw)
    _mld.TransformerDecoderLayerOptimal.forward = _fwd
    _mld.TransformerDecoderLayerOptimal._patched_for_torch2 = True

def make_mld():
    """13 fixed queries, cross-attending a [B, N, 768] token sequence."""
    return MLDecoder(num_classes=len(LABELS), initial_num_features=768,
                     decoder_embedding=768, num_of_groups=len(LABELS))
_m = make_mld()
print('MLDecoder ok |', inspect.signature(MLDecoder.__init__))
print('query_embed learnable:', _m.decoder.query_embed.weight.requires_grad, '(False by design)')
del _m

## C6 · Frozen backbone — chỉ nạp `backbone.*` từ checkpoint 04b

In [ ]:
from transformers import AutoModel
backbone = AutoModel.from_pretrained('microsoft/rad-dino').to(dev).eval()
_sd = torch.load(BEST_CKPT, map_location=dev, weights_only=False)['model']
_bb = {k[len('backbone.'):]: v for k, v in _sd.items() if k.startswith('backbone.')}
assert _bb, 'no backbone.* keys in checkpoint'
backbone.load_state_dict(_bb, strict=True)
for p in backbone.parameters(): p.requires_grad = False
assert sum(p.requires_grad for p in backbone.parameters()) == 0
D = backbone.config.hidden_size; assert D == 768
print(f'frozen backbone: {len(_bb)} tensors | hidden {D} | trainable {sum(p.requires_grad for p in backbone.parameters())}')

@torch.no_grad()
def forward_backbone(images):
    """The ONE forward every head shares. Returns detached CLS + patches."""
    with torch.autocast('cuda', dtype=AMP, enabled=(dev=='cuda')):
        h = backbone(pixel_values=images).last_hidden_state         # [B, 1+P, 768]
    return h[:, 0].detach().float(), h[:, 1:].detach().float()

## C7 · EhrEncoder — copy từ 06·C4, **một instance riêng cho MỖI head**

Nếu H2/H4/H6 dùng chung một instance thì `∇_θE(L2+L4+L6)` trộn ba pipeline và chúng không
còn độc lập. Mỗi head khởi tạo `EhrEncoder` của riêng nó.

In [ ]:
Act = nn.GELU

class EhrEncoder(nn.Module):
    def __init__(self, d_e=128, d_hidden=256, dropout=0.2):
        super().__init__(); T = set(MICRO_TIERS); self.T = T
        self.mods = nn.ModuleDict(); d = 0
        def bag(key, vsize, dim):
            self.mods[key] = nn.EmbeddingBag(vsize, dim, mode='sum'); return dim
        self.set_emb = nn.Embedding(3, 4); d += 4
        d += 2*len(LABS) + 2*len(VITALS) + len(COMO_COLS)
        self.n_vit, self.n_como = len(VITALS), len(COMO_COLS)
        if 1 in T:
            d += bag('t1_org', ORG_V, 16) + bag('t1_spec', SPEC_V, 8)
            self.stat_emb = nn.Embedding(4, 4); d += 4 + 1 + 2*len(REC1)
        if 2 in T: d += bag('t2_spec', SPEC_V, 8) + 3
        if 3 in T:
            d += bag('t3_org', ORG_V, 16); self.rec_emb = nn.Embedding(3, 4)
            d += 4 + 2*len(REC3) + 1
        self.in_dim = d
        self.mlp = nn.Sequential(nn.Linear(d, d_hidden), nn.LayerNorm(d_hidden), Act(),
                                 nn.Dropout(dropout),
                                 nn.Linear(d_hidden, d_e), nn.LayerNorm(d_e), Act())
    def _bag(self, k, b):
        flat, off = b[k]; m = self.mods[k]
        return m(flat, off)
    def forward(self, b):
        T = self.T
        p = [self.set_emb(b['set_id']), b['lab_val'], b['lab_mask']]
        if self.n_vit:  p += [b['vit_val'], b['vit_mask']]
        if self.n_como: p += [b['como']]
        if 1 in T: p += [self._bag('t1_org',b), self._bag('t1_spec',b), self.stat_emb(b['stat_id']),
                         b['t1_same_encounter'][:,None], b['rec1_v'], b['rec1_m']]
        if 2 in T: p += [self._bag('t2_spec',b), b['t2_pending'][:,None],
                         b['t2_time_imprecise'][:,None], b['t2_n_pending'][:,None]]
        if 3 in T: p += [self._bag('t3_org',b), self.rec_emb(b['rec_id']), b['rec3_v'], b['rec3_m'],
                         b['t3_n_specimens'][:,None]]
        return self.mlp(torch.cat(p, 1))
print('EhrEncoder in_dim:', EhrEncoder().in_dim)

## C8 · 8 head, một lớp tham số hóa

`ehr='zero'` chèn một **token hằng số 0** — đây là control cấu trúc, không phải "không có
gì". Lý do nó cần thiết: `nn.MultiheadAttention` có `in_proj_bias`, nên một token input
bằng 0 vẫn cho `k = b_k ≠ 0` và `v = b_v ≠ 0`. Token 0 **không** trung tính; nó là một
token hằng. Vì vậy zero-init `ehr_proj` **không** tạo identity — thay vì cố triệt tiêu
perturbation đó, ta **đo** nó:

- `Δ_structure = H6z − H5` — giá phải trả khi chèn thêm *một token bất kỳ*
- `Δ_content   = H6 − H6z` — giá trị của *nội dung lâm sàng*

`H6 − H5` gộp cả hai và che mất chúng.

Vẫn zero-init `ehr_proj` (giảm nhiễu ngẫu nhiên lúc khởi tạo) và **copy state_dict của
decoder** từ head `none` sang `zero`/`token` cùng nhánh, để ba head chỉ khác nhau ở token
được chèn chứ không khác ở khởi tạo decoder.

In [ ]:
class Arm(nn.Module):
    def __init__(self, cfg: HCfg):
        super().__init__(); self.cfg = cfg
        self.norm = nn.LayerNorm(cfg.d_cxr)                       # on CLS or on patches
        if cfg.ehr in ('concat','token'):
            self.enc = EhrEncoder(cfg.d_e, cfg.d_hidden, cfg.dropout_ehr)
        if cfg.ehr == 'token':
            self.ehr_proj = nn.Linear(cfg.d_e, cfg.d_cxr)
            nn.init.zeros_(self.ehr_proj.weight); nn.init.zeros_(self.ehr_proj.bias)  # not identity; see md
            self.ehr_norm = nn.LayerNorm(cfg.d_cxr)
            self.ehr_drop = nn.Dropout(cfg.dropout_ehr)
        if cfg.ehr in ('token','zero'):
            self.img_type = nn.Parameter(torch.zeros(1,1,cfg.d_cxr))
            self.ehr_type = nn.Parameter(torch.zeros(1,1,cfg.d_cxr))
        if cfg.head == 'mlp':
            d_in = cfg.d_cxr + (cfg.d_e if cfg.ehr == 'concat' else 0)
            self.mlp = nn.Sequential(nn.Linear(d_in, cfg.d_hidden), nn.LayerNorm(cfg.d_hidden),
                                     Act(), nn.Dropout(cfg.dropout_head),
                                     nn.Linear(cfg.d_hidden, len(LABELS)))
        else:
            self.mld = make_mld()

    def tokens(self, cls, patches, batch):
        """Build the [B, S, 768] sequence this arm feeds to ML-Decoder."""
        x = self.norm(cls[:, None, :] if self.cfg.repr == 'cls' else patches)
        if self.cfg.ehr == 'none': return x
        x = x + self.img_type
        if self.cfg.ehr == 'zero':
            t = torch.zeros(x.shape[0], 1, x.shape[2], device=x.device, dtype=x.dtype)
        else:
            e = self.enc(batch)
            t = self.ehr_drop(self.ehr_norm(self.ehr_proj(e)))[:, None, :]
        return torch.cat([x, t + self.ehr_type], 1)

    def forward(self, cls, patches, batch):
        if self.cfg.head == 'mlp':
            f = self.norm(cls)
            if self.cfg.ehr == 'concat': f = torch.cat([f, self.enc(batch)], 1)
            return self.mlp(f)
        return self.mld(self.tokens(cls, patches, batch))

def build_arms(seed):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    arms = {c.tag: Arm(c).to(dev) for c in HEADS}
    # same decoder init within a branch -> the arms differ only by the inserted token
    for base, others in (('H3', ('H4z','H4')), ('H5', ('H6z','H6'))):
        for o in others: arms[o].mld.load_state_dict(arms[base].mld.state_dict())
    return arms
_a = build_arms(42)
print('trainable params per arm:')
for t, m in _a.items():
    print(f'  {t:4s} {sum(p.numel() for p in m.parameters() if p.requires_grad)/1e6:6.2f} M')
del _a

## C9 · ASL — port chuẩn (bản tự chế đã làm sụp D5_asl xuống 0.2778)

Dùng **cùng một loss cho cả 8 head** (§2.2 của plan). ASL thắng cả ba tier ở `06`
(head/medium/tail) sau khi port đúng.

In [ ]:
class ASLoss(nn.Module):
    """Asymmetric Loss (Ben-Baruch 2021), Alibaba-MIIL/ASL, math verbatim."""
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, eps=1e-8):
        super().__init__(); self.gn, self.gp, self.clip, self.eps = gamma_neg, gamma_pos, clip, eps
    def forward(self, x, y):
        y = y.float(); xs_pos = torch.sigmoid(x); xs_neg = 1.0 - xs_pos
        if self.clip: xs_neg = (xs_neg + self.clip).clamp(max=1)
        los = y*torch.log(xs_pos.clamp(min=self.eps)) + (1-y)*torch.log(xs_neg.clamp(min=self.eps))
        pt = xs_pos*y + xs_neg*(1-y); g = self.gp*y + self.gn*(1-y)
        return -(los*torch.pow(1-pt, g)).mean()

with torch.no_grad():
    _a = ASLoss()
    _e = _a(torch.tensor([[-4.6]]), torch.tensor([[0.0]])).item()
    _h = _a(torch.tensor([[ 4.6]]), torch.tensor([[0.0]])).item()
assert _e < 1e-3 and _h > 1.0, (_e, _h)
print(f'ASL check: easy-neg {_e:.5f} (~0) | hard-neg {_h:.3f} (large)  OK')

## C10 · Predict ở mức ảnh → pool 7:3 ở mức study

Mẫu số renormalize: 51.6% study chỉ có frontal, tổng-có-trọng-số sẽ nhân chúng với 0.7.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

@torch.no_grad()
def predict_images(arms, split, bs, perm=None):
    """One shared backbone pass; every arm scores the same images. -> {tag: [n_img, 13]}"""
    for m in arms.values(): m.eval()
    P = {t: np.zeros((len(IDX[split]), len(LABELS)), 'float32') for t in arms}
    pos = 0
    for b in tqdm(loader(split, eval_tf, False, bs, perm, workers=0),   # 0 -> no teardown spam
                  desc=f'predict[{split}]', leave=False):
        b = to_dev(b); cls, patches = forward_backbone(b['image']); n = cls.shape[0]
        for t, m in arms.items():
            P[t][pos:pos+n] = torch.sigmoid(m(cls, patches, b)).float().cpu().numpy()
        pos += n
    return P

def pool_study(p_img, split):
    """p_study = (w_f*mean_f + w_l*mean_l) / (w_f*[has_f] + w_l*[has_l])."""
    sub = man.iloc[IDX[split]]
    d = pd.DataFrame(p_img, columns=LABELS)
    d['study_id'] = sub.study_id.to_numpy(); d['vg'] = sub.view_group.to_numpy()
    sid = np.sort(d.study_id.unique())
    fr = d[d.vg=='frontal'].groupby('study_id')[LABELS].mean().reindex(sid)
    la = d[d.vg=='lateral'].groupby('study_id')[LABELS].mean().reindex(sid)
    hf = fr.notna().all(1).to_numpy().astype('float32'); hl = la.notna().all(1).to_numpy().astype('float32')
    den = (W_F*hf + W_L*hl)[:, None]; assert (den > 0).all()
    PS = (W_F*np.nan_to_num(fr.to_numpy('float32'))*hf[:,None]
          + W_L*np.nan_to_num(la.to_numpy('float32'))*hl[:,None]) / den
    return sid, PS

def ap_study(p_img, split):
    sid, PS = pool_study(p_img, split)
    Yt = ehr.set_index('study_id').loc[sid, LABELS].to_numpy('float32')
    return np.array([average_precision_score(Yt[:,j], PS[:,j]) for j in range(len(LABELS))]), sid, PS

## C11 · Vòng train dùng chung forward — 8 optimizer độc lập

Backward **tuần tự** trên feature đã `detach()`. So với cộng loss rồi backward một lần,
cách này: không giữ 8 graph cùng lúc, clip **riêng** từng head (clip global sẽ scale mọi
head theo một norm chung ⇒ head gradient lớn bóp head khác ⇒ mất tính tương đương), và bỏ
qua được head đã early-stop.

Batch **không khóa cứng** — `probe_batch()` tăng dần rồi chừa ~20% VRAM. Ước lượng chỉ từ
tensor patch bỏ sót attention workspace, Q/K/V, ảnh gốc và fragmentation.

`accum = EFFECTIVE_BATCH // actual` ⇒ mọi head thấy **cùng effective batch 256**.
Scheduler đếm theo **optimizer step**, không theo iteration.

In [ ]:
def probe_batch(cands=(32,48,64,96,128)):
    if dev != 'cuda': return 8
    arms = build_arms(0); ok = cands[0]
    for B in cands:
        try:
            torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
            b = to_dev(collate([ImgDS(IDX['train'], train_tf)[k] for k in range(B)]))
            cls, patches = forward_backbone(b['image'])
            for m in arms.values():
                loss = ASLoss()(m(cls, patches, b), b['y']); loss.backward()
            peak = torch.cuda.max_memory_allocated()/1e9
            tot  = torch.cuda.get_device_properties(0).total_memory/1e9
            print(f'  B={B:4d}  peak {peak:5.1f} GB / {tot:.0f} GB  ({peak/tot:5.1%})')
            if peak/tot < 0.80: ok = B
            else: break
        except torch.cuda.OutOfMemoryError:
            print(f'  B={B:4d}  OOM'); break
        finally:
            for m in arms.values(): m.zero_grad(set_to_none=True)
    del arms; torch.cuda.empty_cache()
    return ok

def param_groups(m, wd):
    dec = [p for p in m.parameters() if p.requires_grad and p.ndim > 1]
    nod = [p for p in m.parameters() if p.requires_grad and p.ndim <= 1]
    return [{'params':dec,'weight_decay':wd},{'params':nod,'weight_decay':0.0}]

def cosine_warmup(opt, total, warm):
    f = lambda s: s/max(warm,1) if s < warm else 0.5*(1+math.cos(math.pi*(s-warm)/max(total-warm,1)))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)

class Exp:
    """One arm + its OWN optimizer / scheduler / clip / early-stop. Nothing shared."""
    def __init__(self, tag, model, total):
        self.tag, self.model = tag, model
        self.crit = ASLoss()
        self.opt  = torch.optim.AdamW(param_groups(model, WD), lr=LR)
        self.sch  = cosine_warmup(self.opt, total, int(WARMUP*total))
        self.best, self.since, self.done, self.best_sd = -1.0, 0, False, None

def ck_resume(seed): return f'{TF_DIR}/checkpoints/_resume_s{seed}.pt'

def save_resume(seed, exps, ep):
    """Atomic .tmp -> os.replace: a disconnect DURING the write cannot leave a half file
    that the next run would happily load. Same rule as 04a's shards and 04b's checkpoints."""
    obj = {'epoch': ep, 'batch': batch_of[seed], 'epochs': epochs_of[seed],
           'arms': {t: dict(model=e.model.state_dict(), opt=e.opt.state_dict(),
                            sch=e.sch.state_dict(), best=e.best, since=e.since,
                            done=e.done, best_sd=e.best_sd)
                    for t, e in exps.items()}}
    p = ck_resume(seed); torch.save(obj, p + '.tmp'); os.replace(p + '.tmp', p)

batch_of, epochs_of = {}, {}      # what each seed's resume file was built with

def train_shared(seed, batch, epochs=EPOCHS, verbose=True):
    arms = build_arms(seed)
    dl = loader('train', train_tf, True, batch)
    accum = max(1, EFFECTIVE_BATCH // batch)
    total = epochs * math.ceil(len(dl)/accum)
    exps  = {t: Exp(t, m, total) for t, m in arms.items()}
    if verbose: print(f'seed {seed} | batch {batch} x accum {accum} = {batch*accum} effective | '
                      f'{len(dl)} it/epoch | {total} opt-steps')

    # ---- resume: 8 arms x 2.5h is far too much to lose to one Colab disconnect ----
    # The scheduler state is restored too: without it OneCycle/cosine would restart from
    # step 0, the LR would jump back to its peak mid-training and wreck the trajectory.
    batch_of[seed], epochs_of[seed] = batch, epochs
    start = 0
    if os.path.exists(ck_resume(seed)):
        c = torch.load(ck_resume(seed), map_location=dev, weights_only=False)
        # The scheduler state is only meaningful for the (batch, epochs) it was built with:
        # total_steps = epochs * ceil(len(dl)/accum). Restore it under a different batch and
        # the cosine curve silently lands on the wrong LR for the rest of training. Refuse.
        if (c.get('batch'), c.get('epochs')) != (batch, epochs):
            raise RuntimeError(
                f"resume file for seed {seed} was built with batch={c.get('batch')}, "
                f"epochs={c.get('epochs')} but you are running batch={batch}, epochs={epochs}.\n"
                f"    The LR schedule would be wrong. Either use the old values, or delete\n"
                f"    {ck_resume(seed)} and start this seed over.")
        for t, e in exps.items():
            a = c['arms'][t]
            e.model.load_state_dict(a['model']); e.opt.load_state_dict(a['opt'])
            e.sch.load_state_dict(a['sch'])
            e.best, e.since, e.done, e.best_sd = a['best'], a['since'], a['done'], a['best_sd']
        start = c['epoch'] + 1
        if verbose:
            print(f'RESUMED seed {seed} at epoch {start}/{epochs}  ' +
                  ' '.join(f'{t}:{e.best:.4f}{"(done)" if e.done else ""}' for t, e in exps.items()))
    if start >= epochs or all(e.done for e in exps.values()):
        for t, e in exps.items(): e.model.load_state_dict(e.best_sd)
        if verbose: print(f'seed {seed} already complete -- skipping')
        return {t: e.model for t, e in exps.items()}, {t: e.best for t, e in exps.items()}

    for ep in range(start, epochs):
        for m in arms.values(): m.train()
        run = {t: 0.0 for t in exps}
        for i, b in enumerate(tqdm(dl, desc=f's{seed} ep{ep+1}/{epochs}', leave=False)):
            b = to_dev(b)
            cls, patches = forward_backbone(b['image'])        # THE shared forward
            for t, e in exps.items():
                if e.done: continue
                loss = e.crit(e.model(cls, patches, b), b['y']) / accum
                loss.backward()                                # graph is per-arm; features detached
                run[t] += loss.item()*accum
                if (i+1) % accum == 0 or i == len(dl)-1:
                    nn.utils.clip_grad_norm_(e.model.parameters(), CLIP)   # PER ARM, never global
                    e.opt.step(); e.sch.step(); e.opt.zero_grad(set_to_none=True)
        Pv = predict_images({t:e.model for t,e in exps.items() if not e.done}, 'val', batch*2)
        line = []
        for t, e in exps.items():
            if e.done: line.append(f'{t}:done'); continue
            mAP = ap_study(Pv[t], 'val')[0].mean()
            if mAP > e.best:
                e.best, e.since = mAP, 0
                e.best_sd = {k: v.detach().cpu().clone() for k, v in e.model.state_dict().items()}
            else:
                e.since += 1
                if e.since >= PATIENCE: e.done = True
            line.append(f'{t}:{mAP:.4f}{"*" if e.since==0 else ""}')
        save_resume(seed, exps, ep)                # EVERY epoch -- this is the resume point
        if verbose: print(f'  ep{ep+1} val ' + ' '.join(line) + '  [resume saved]')
        if all(e.done for e in exps.values()):
            if verbose: print('  all arms early-stopped'); break
    for t, e in exps.items(): e.model.load_state_dict(e.best_sd)
    return {t: e.model for t, e in exps.items()}, {t: e.best for t, e in exps.items()}

## C12 · Cổng kiểm tra — chạy TRƯỚC khi tốn GPU

1. **Backbone identity** — mọi arm phải nhận **cùng** feature (đó là toàn bộ tiền đề của
   shared forward).
2. **Gradient không rò vào backbone**, và có chảy vào đúng module của từng arm.
3. **Shapes** — `[B,768]`, `[B,1369,768]`, `[B,128]`, seq 1/2/1369/1370, logits `[B,13]`.
4. **`H6z` chỉ khác `H5` đúng một token** — nếu decoder init khác nhau thì
   `Δ_structure` đo nhầm sang khởi tạo.
5. **EHR alignment** — `study_id` của ảnh khớp `ehr_row`; nhãn **không** đi theo EHR khi shuffle.

In [ ]:
_b = to_dev(collate([ImgDS(IDX['train'], eval_tf)[k] for k in range(4)]))
arms = build_arms(42)

# 1 -- backbone identity
c1, p1 = forward_backbone(_b['image']); c2, p2 = forward_backbone(_b['image'])
assert torch.allclose(c1, c2) and torch.allclose(p1, p2), 'backbone not deterministic in eval'
print(f'G1 backbone identity OK | cls {tuple(c1.shape)} patches {tuple(p1.shape)}')
assert p1.shape[1] == 1369 and c1.shape[1] == 768

# 3 -- shapes per arm + sequence length
for t, m in arms.items():
    m.eval()
    with torch.no_grad():
        o = m(c1, p1, _b)
        s = m.tokens(c1, p1, _b).shape[1] if m.cfg.head == 'mld' else None
    assert o.shape == (4, 13), (t, o.shape)
    print(f'G3 {t:4s} logits {tuple(o.shape)}' + (f' | seq {s}' if s else ' | mlp'))
exp_seq = {'H3':1, 'H4z':2, 'H4':2, 'H5':1369, 'H6z':1370, 'H6':1370}
for t, n in exp_seq.items():
    with torch.no_grad(): assert arms[t].tokens(c1, p1, _b).shape[1] == n, (t, n)
print('G3 sequence lengths OK')

# 4 -- H6z differs from H5 by exactly one token (same decoder init)
for a, b in (('H5','H6z'), ('H3','H4z')):
    sa, sb = arms[a].mld.state_dict(), arms[b].mld.state_dict()
    assert all(torch.equal(sa[k], sb[k]) for k in sa), f'{a}/{b} decoder init differs'
print('G4 zero-token arms share the decoder init with their none-arm  OK')

# 2 -- gradients
for m in arms.values(): m.train(); m.zero_grad(set_to_none=True)
loss = ASLoss()(arms['H6'](c1, p1, _b), _b['y']); loss.backward()
assert all(p.grad is None for p in backbone.parameters()), 'GRADIENT LEAKED INTO BACKBONE'
assert any(p.grad is not None for p in arms['H6'].mld.parameters())
assert any(p.grad is not None for p in arms['H6'].enc.parameters())
assert any(p.grad is not None for p in arms['H6'].ehr_proj.parameters())
assert all(p.grad is None for p in arms['H5'].parameters()), 'H5 got gradient from H6 loss -> arms share params!'
print('G2 gradients: backbone clean · H6 modules got grad · H5 untouched by H6 loss  OK')

# 5 -- EHR alignment
for k in np.random.default_rng(0).choice(len(man), min(200, len(man)), replace=False):
    assert int(ehr.study_id.iloc[int(man.ehr_row.iloc[k])]) == int(man.study_id.iloc[k])
g = man.groupby('study_id').ehr_row.nunique()
assert g.max() == 1, 'images of one study point at different EHR rows'
print('G5 EHR alignment OK (200 samples + every study has exactly 1 ehr_row)')
for m in arms.values(): m.zero_grad(set_to_none=True)
del arms; torch.cuda.empty_cache() if dev=='cuda' else None

## C13 · Pilot seed 42 — chỉ nhìn **validation**

§7.1: không báo test trong giai đoạn chọn hyperparameter. Cell này để kiểm memory, chọn
batch, xem learning curve, và xác nhận EHR token không làm training sụp.

In [ ]:
print('probing batch...'); BATCH = probe_batch(); print('-> BATCH =', BATCH)
t0 = time.time()
models_pilot, best_pilot = train_shared(42, BATCH)
print(f'\npilot done in {(time.time()-t0)/60:.1f} min')
print('\nval macro mAP (pilot, seed 42):')
for t in [h.tag for h in HEADS]: print(f'  {t:4s} {best_pilot[t]:.4f}')
print('\nsanity: H6 phai >= H5 khong? chua chac -- do la ket qua, khong phai bug.')

## C14 · Ba seed — khóa schedule rồi mới chạy

In [ ]:
RUN_SEEDS = True
AP_TEST, AP_VAL = {}, {}
if RUN_SEEDS:
    for s in SEEDS:
        f = f'{TF_DIR}/metrics/ap_test_s{s}.npz'
        if os.path.exists(f):
            z = np.load(f); AP_TEST[s] = {t: z[t] for t in z.files}
            print(f'seed {s}: loaded from cache'); continue
        models, best = train_shared(s, BATCH)
        Pte = predict_images(models, 'test', BATCH*2)
        AP_TEST[s] = {t: ap_study(Pte[t], 'test')[0] for t in models}
        AP_VAL[s]  = best
        np.savez(f, **AP_TEST[s])
        for t, m in models.items():
            torch.save({'model': m.state_dict(), 'cfg': vars(m.cfg), 'val_mAP': best[t]},
                       f'{TF_DIR}/checkpoints/{t}_s{s}_best.pt')
        pd.DataFrame(Pte).to_parquet(f'{TF_DIR}/predictions/image_pred_s{s}.parquet') \
            if False else None
        print(f'seed {s} test macro: ' + ' '.join(f'{t}:{AP_TEST[s][t].mean():.4f}' for t in models))
    print('\ndone', len(AP_TEST), 'seeds')

## C15 · Delta + interaction — **paired theo seed**

Paired vì cùng seed = cùng khởi tạo, cùng data order, cùng ảnh, cùng feature. Welch độc
lập (như `06/C12` dùng) vứt bỏ đúng cái pairing đó.

**PRIMARY là `I_content`.** Mọi số khác là secondary/exploratory — nêu trước để không rơi
vào bẫy multiple-comparisons đã giết giả thuyết Pneumonia.

In [ ]:
from scipy import stats as sstats
def ci95(v):
    v = np.asarray(v, float); n = len(v)
    return (float(v.mean()), 0.0) if n < 2 else \
           (float(v.mean()), float(sstats.t.ppf(0.975, n-1)*v.std(ddof=1)/np.sqrt(n)))
M = lambda t: np.array([AP_TEST[s][t].mean() for s in sorted(AP_TEST)])   # per-seed macro

print('===== macro mAP theo seed (mean +/- 95% CI) =====')
print(pd.DataFrame([dict(head=h.tag, **dict(zip(('mean','ci95'), np.round(ci95(M(h.tag)), 4))))
                    for h in HEADS]).to_string(index=False))

def paired(name, vec):
    mu, h = ci95(vec)
    t, p = sstats.ttest_1samp(vec, 0.0)
    print(f'  {name:34s} {mu:+.4f} +/-{h:.4f}  p={p:.3f}  '
          f'{"distinguishable" if p < 0.05 else "NOT distinguishable"}')

print('\n===== PRIMARY =====')
I_content = (M('H6') - M('H6z')) - (M('H4') - M('H4z'))
paired('I_content = (H6-H6z)-(H4-H4z)', I_content)

print('\n===== SECONDARY (bao cao, khong claim) =====')
paired('D_structure  = H6z - H5',        M('H6z') - M('H5'))
paired('D_content    = H6  - H6z',       M('H6')  - M('H6z'))
paired('D_structure' + ' (cls) = H4z-H3', M('H4z') - M('H3'))
paired('D_content   (cls) = H4 - H4z',   M('H4')  - M('H4z'))
paired('I_repr     = (H6-H5)-(H4-H3)',   (M('H6')-M('H5')) - (M('H4')-M('H3')))
paired('I_pipeline = (H6-H5)-(H2-H1)',   (M('H6')-M('H5')) - (M('H2')-M('H1')))
paired('EHR gain @cls  = H2 - H1',       M('H2') - M('H1'))
paired('EHR gain @patch= H6 - H5',       M('H6') - M('H5'))
paired('repr gain      = H5 - H1',       M('H5') - M('H1'))

print('\n--- doc ket qua ---')
mu, h = ci95(I_content)
if mu - h > 0:   print('  I_content > 0: EHR dong gop NHIEU HON tren patch -> ung ho token-level fusion.')
elif mu + h < 0: print('  I_content < 0: EHR dong gop IT HON tren patch -> gain cu chu yeu va bottleneck CLS.')
else:            print('  I_content ~ 0 (CI chua 0): EHR dong gop DOC LAP voi representation, HOAC 3 seed khong du power.')

## C16 · real / zero / shuffle — bằng chứng **nhân quả** chính

Attention mass (C17) là chỉ dấu **cơ chế**; cái quyết định là can thiệp.

**Shuffle phải ở mức `study_id`**, trong strata `(split, setting)`: mọi ảnh của một study
nhận **cùng** EHR đã tráo, nếu không một study mang nhiều clinical context khác nhau trước
khi pool 7:3. Và **nhãn không bao giờ đi theo EHR** — `y` luôn lấy từ `ehr_row` thật.

In [ ]:
def study_perm(seed=0):
    """Permute EHR rows within (split, setting). Row-level identity => every image of a
    study follows the same permuted study, because ehr_row is per-study."""
    rng = np.random.default_rng(seed); perm = np.arange(N)
    key = ehr.split.astype(str) + '|' + ehr.setting.astype(str).str.lower()
    for k in key.unique():
        idx = np.where(key.to_numpy() == k)[0]
        perm[idx] = rng.permutation(idx)
    moved = (perm != np.arange(N)).mean()
    return perm, moved

PERM, moved = study_perm(0)
print(f'shuffle: {moved:.1%} studies got a different EHR row (within split x setting)')

DIAG = {}
s0 = sorted(AP_TEST)[0]
arms_d = {}
for h in HEADS:
    p = f'{TF_DIR}/checkpoints/{h.tag}_s{s0}_best.pt'
    if not os.path.exists(p): continue
    m = Arm(h).to(dev); m.load_state_dict(torch.load(p, map_location=dev, weights_only=False)['model'])
    arms_d[h.tag] = m.eval()

for mode, perm in (('real', None), ('shuffle', PERM)):
    P = predict_images(arms_d, 'test', BATCH*2, perm=perm)
    for t in arms_d: DIAG.setdefault(t, {})[mode] = ap_study(P[t], 'test')[0].mean()
# 'zero' = mask every EHR input -> reuse the zero-token arms as the natural control
for t in arms_d:
    DIAG[t]['zero_arm'] = {'H2':'H1','H4':'H4z','H6':'H6z'}.get(t, '-')

d = pd.DataFrame([dict(head=t, real=round(v['real'],4), shuffle=round(v['shuffle'],4),
                       delta=round(v['real']-v['shuffle'],4), zero_ref=v['zero_arm'])
                  for t, v in DIAG.items()])
print('\n' + d.to_string(index=False))
print('\n  real > shuffle => model dung EHR CUA DUNG BENH NHAN, khong phai chi them capacity.')
print('  real ~ shuffle => token chi la learned bias / distributional shortcut.')
pd.DataFrame(DIAG).T.to_csv(f'{TF_DIR}/metrics/ehr_shuffle_diagnostics.csv')

## C17 · Attention mass + value-weighted contribution

**Không phải bằng chứng nhân quả.** Attention weight nhỏ + value norm lớn vẫn có thể đóng
góp đáng kể; weight lớn không bảo đảm output phụ thuộc token đó. Đây là **chỉ dấu cơ
chế**, đọc kèm C16.

Lấy weight mà **không phải vá `ml_decoder.py`**: layer gọi
`self.multihead_attn(tgt, memory, memory)[0]` — weight *vẫn được tính*, chỉ bị vứt. Ta bọc
`forward` để ép `average_attn_weights=False` (per-head, như review yêu cầu) và bắt lại.
Chỉ bật lúc chẩn đoán ⇒ không đụng tốc độ train.

Contribution được **verify bằng tái dựng**: nếu tổng đóng góp của mọi token dựng lại đúng
output thật của MHA thì phép phân rã là đúng — không phải "tin là đúng".

In [ ]:
class AttnCapture:
    def __init__(self, mha): self.mha, self.w, self.mem = mha, None, None
    def __enter__(self):
        self._o = self.mha.forward
        def fwd(q, k, v, **kw):
            kw['need_weights'] = True; kw['average_attn_weights'] = False
            out = self._o(q, k, v, **kw)
            self.w = out[1].detach()        # [B, nhead, n_query, S]
            self.mem = k.detach()           # [S, B, E]  (seq-first)
            return out
        self.mha.forward = fwd; return self
    def __exit__(self, *e): self.mha.forward = self._o

@torch.no_grad()
def attn_report(tag, n_batch=4):
    m = arms_d[tag]; m.eval()
    mha = m.mld.decoder.layers[0].multihead_attn
    E = mha.embed_dim; H = mha.num_heads; dh = E // H
    W_v = mha.in_proj_weight[2*E:3*E]; b_v = mha.in_proj_bias[2*E:3*E]
    mass, ratio = [], []
    dl = loader('test', eval_tf, False, BATCH)
    for i, b in enumerate(dl):
        if i >= n_batch: break
        b = to_dev(b); cls, patches = forward_backbone(b['image'])
        with AttnCapture(mha) as cap: m(cls, patches, b)
        a   = cap.w                                   # [B, H, Q, S]
        mem = cap.mem                                 # [S, B, E]
        S, B, _ = mem.shape
        v = F.linear(mem, W_v, b_v).view(S, B, H, dh).permute(1, 2, 0, 3)   # [B,H,S,dh]
        # contribution of each token = a * v, concat heads, then out_proj
        ctx_all = torch.einsum('bhqs,bhsd->bhqd', a, v)                     # [B,H,Q,dh]
        out_all = mha.out_proj(ctx_all.permute(0,2,1,3).reshape(B, -1, E))  # [B,Q,E]
        ctx_E   = torch.einsum('bhq,bhd->bhqd', a[..., -1], v[:, :, -1, :]) # EHR token only
        out_E   = mha.out_proj(ctx_E.permute(0,2,1,3).reshape(B, -1, E))
        # VERIFY the decomposition reproduces the real attention output
        if i == 0:
            ref = torch.einsum('bhqs,bhsd->bhqd', a, v).permute(0,2,1,3).reshape(B, -1, E)
            assert torch.allclose(mha.out_proj(ref), out_all, atol=1e-4), 'decomposition wrong'
        mass.append(a[..., -1].mean(1).cpu())                               # [B,Q] avg over heads
        ratio.append((out_E.norm(dim=-1) / (out_all.norm(dim=-1) + 1e-8)).cpu())
    mass = torch.cat(mass).mean(0).numpy(); ratio = torch.cat(ratio).mean(0).numpy()
    S = 1370 if arms_d[tag].cfg.repr == 'patch' else 2
    r = pd.DataFrame({'label': LABELS, 'tier': TIER, 'attn_mass': mass.round(5),
                      'vs_uniform': (mass*S).round(2), 'value_ratio': ratio.round(4)})
    print(f'\n=== {tag} === uniform = 1/{S} = {1/S:.5f}')
    print(r.sort_values('attn_mass', ascending=False).to_string(index=False))
    print(f'  mean attn on EHR token: {mass.mean():.5f} ({mass.mean()*S:.2f}x uniform)')
    print(f'  mean ||contribution_EHR|| / ||contribution_all||: {ratio.mean():.4f}')
    return r

for t in ('H6', 'H4'):
    if t in arms_d: attn_report(t).to_csv(f'{TF_DIR}/metrics/attn_{t}.csv', index=False)
print('\n  attn ~ uniform NHUNG value_ratio cao => token duoc dung du attention nho.')
print('  ca hai deu thap => class queries khong nhin EHR token -> fusion hong VE CO CHE,')
print('  khong phai "EHR vo dung". Doc cung C16.')

## C18 · Subject-level bootstrap — CI không dựa vào 3 seed

3 seed ⇒ 2 bậc tự do ⇒ CI rất rộng. Bootstrap trên test predictions dùng **31,669 study /
~25k subject** nên chặt hơn nhiều. Đơn vị lấy mẫu là **subject**, không phải study: nhiều
study của cùng bệnh nhân **không** độc lập.

Hai CI đo hai thứ khác nhau và đều cần: seed-CI = biến thiên do khởi tạo; bootstrap-CI =
biến thiên do mẫu test.

In [ ]:
N_BOOT = 2000
s0 = sorted(AP_TEST)[0]
P_te = predict_images(arms_d, 'test', BATCH*2)
sid, _ = pool_study(P_te['H6'], 'test')
PS = {t: pool_study(P_te[t], 'test')[1] for t in arms_d}
Yt = ehr.set_index('study_id').loc[sid, LABELS].to_numpy('float32')
subj_of = ehr.set_index('study_id').loc[sid, 'subject_id'].to_numpy()
subjects = np.unique(subj_of)
by_subj = {s: np.where(subj_of == s)[0] for s in subjects}
print(f'bootstrap: {len(sid):,} studies / {len(subjects):,} subjects, {N_BOOT} resamples')

def macro(Ps, rows):
    y = Yt[rows]
    keep = [j for j in range(len(LABELS)) if y[:, j].sum() > 0]
    return np.mean([average_precision_score(y[:, j], Ps[rows][:, j]) for j in keep])

rng = np.random.default_rng(0); boot = []
for _ in tqdm(range(N_BOOT), desc='bootstrap'):
    pick = rng.choice(subjects, len(subjects), replace=True)
    rows = np.concatenate([by_subj[s] for s in pick])
    boot.append(((macro(PS['H6'], rows) - macro(PS['H6z'], rows))
                 - (macro(PS['H4'], rows) - macro(PS['H4z'], rows))))
boot = np.array(boot)
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f'\nI_content (subject bootstrap): {boot.mean():+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]')
print('  CI khong chua 0 ->' if lo*hi > 0 else '  CI chua 0 ->',
      'phan biet duoc' if lo*hi > 0 else 'khong phan biet duoc')
pd.DataFrame({'I_content': boot}).to_csv(f'{TF_DIR}/metrics/interaction_bootstrap.csv', index=False)
json.dump({'heads': [vars(h) for h in HEADS], 'seeds': SEEDS, 'batch': int(BATCH),
           'effective_batch': EFFECTIVE_BATCH, 'epochs': EPOCHS, 'patience': PATIENCE,
           'backbone': BEST_CKPT, 'note': 'backbone = 04b FT-then-frozen, not raw pretrained'},
          open(f'{TF_DIR}/experiment_manifest.json', 'w'), indent=1)
print('\nDONE. artifacts in', TF_DIR)